### CRISP-DM · Fase 3: Preparación de los datos

Limpieza y definición del preprocesador. Las reglas se definen con el grupo de entrenamiento y se aplican igual al de prueba.

- **Entradas:** `crisp_dm/data/interim/train.csv`, `crisp_dm/data/interim/test.csv`
- **Salidas:**
  - `crisp_dm/data/interim/train_clean.csv`, `crisp_dm/data/interim/test_clean.csv`: datos limpios, sin escalar
  - `crisp_dm/data/processed/train_prepared.csv`, `crisp_dm/data/processed/test_prepared.csv`: datos transformados
  - `crisp_dm/models/preprocessor.joblib`: preprocesador ajustado

#### Librerías

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#### Rutas

Relativas a la carpeta del notebook.

In [2]:
INTERIM_DIR = Path("../data/interim")
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
for folder in [PROCESSED_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

#### Carga de datos

In [3]:
train = pd.read_csv(INTERIM_DIR / "train.csv")
test = pd.read_csv(INTERIM_DIR / "test.csv")

TARGET = "species"
NUMERIC = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
CATEGORICAL = ["island", "sex"]
FEATURES = NUMERIC + CATEGORICAL

print(f"Entrenamiento: {train.shape} | Prueba: {test.shape}")

Entrenamiento: (275, 8) | Prueba: (69, 8)


#### Eliminación de registros sin medidas

In [4]:
removed = train.loc[train[NUMERIC].isna().all(axis=1), "row_id"].tolist()
removed_test = test.loc[test[NUMERIC].isna().all(axis=1), "row_id"].tolist()

train = train[train[NUMERIC].notna().any(axis=1)].copy()
test = test[test[NUMERIC].notna().any(axis=1)].copy()

print(f"Eliminados en entrenamiento: {removed}")
print(f"Eliminados en prueba: {removed_test}")

Eliminados en entrenamiento: [3, 339]
Eliminados en prueba: []


#### Tratamiento de `sex`

Todo valor distinto de `MALE` o `FEMALE` pasa a `UNKNOWN`.

In [5]:
VALID_SEX = ["MALE", "FEMALE"]

for df in (train, test):
    df["sex"] = df["sex"].where(df["sex"].isin(VALID_SEX), "UNKNOWN")

pd.DataFrame({
    "train": train["sex"].value_counts(),
    "test": test["sex"].value_counts(),
}).fillna(0).astype(int)

,train,test
sex,,
FEMALE,134,31
MALE,133,35
UNKNOWN,6,3


#### Verificación de datos limpios

In [6]:
pd.DataFrame({
    "faltantes_train": train[FEATURES + [TARGET]].isna().sum(),
    "faltantes_test": test[FEATURES + [TARGET]].isna().sum(),
})

,faltantes_train,faltantes_test
culmen_length_mm,0,0
culmen_depth_mm,0,0
flipper_length_mm,0,0
body_mass_g,0,0
island,0,0
sex,0,0
species,0,0


#### Preprocesador

- Numéricas: imputación por mediana y estandarización; SVM es sensible a la escala de las variables.
- Categóricas: codificación one-hot; una categoría no vista se codifica como ceros.

In [7]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
])
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


#### Ajuste y transformación

El preprocesador se ajusta solo con el grupo de entrenamiento.

In [8]:
preprocessor.fit(train[FEATURES])
columns = preprocessor.get_feature_names_out().tolist()


def prepare(df):
    prepared = pd.DataFrame(preprocessor.transform(df[FEATURES]), columns=columns, index=df.index)
    return pd.concat([df[["row_id", TARGET]], prepared], axis=1)


train_prepared = prepare(train)
test_prepared = prepare(test)
train_prepared.head()

,row_id,species,num__culmen_length_mm,num__culmen_depth_mm,num__flipper_length_mm,num__body_mass_g,cat__island_Biscoe,cat__island_Dream,cat__island_Torgersen,cat__sex_FEMALE,cat__sex_MALE,cat__sex_UNKNOWN
0,193,Chinstrap,0.440701,0.176677,-0.964733,-0.663196,0.0,1.0,0.0,1.0,0.0,0.0
1,98,Adelie,-1.970312,-0.528187,-1.605714,-1.587406,0.0,1.0,0.0,1.0,0.0,0.0
2,155,Chinstrap,0.293464,0.780846,-0.893513,-0.817231,0.0,1.0,0.0,1.0,0.0,0.0
3,40,Adelie,-1.344553,0.428414,-1.320833,-1.279336,0.0,1.0,0.0,1.0,0.0,0.0
4,281,Gentoo,0.440701,-1.132355,1.456751,1.370064,1.0,0.0,0.0,0.0,1.0,0.0


#### Verificación de la transformación

In [9]:
num_cols = [c for c in columns if c.startswith("num__")]
pd.DataFrame({
    "media_train": train_prepared[num_cols].mean(),
    "desv_train": train_prepared[num_cols].std(ddof=0),
    "media_test": test_prepared[num_cols].mean(),
    "desv_test": test_prepared[num_cols].std(ddof=0),
}).round(3)

,media_train,desv_train,media_test,desv_test
num__culmen_length_mm,0.0,1.0,0.106,1.012
num__culmen_depth_mm,-0.0,1.0,0.005,0.964
num__flipper_length_mm,0.0,1.0,0.130,0.993
num__body_mass_g,-0.0,1.0,0.083,0.930


#### Guardado de resultados

In [10]:
train.to_csv(INTERIM_DIR / "train_clean.csv", index=False)
test.to_csv(INTERIM_DIR / "test_clean.csv", index=False)
train_prepared.to_csv(PROCESSED_DIR / "train_prepared.csv", index=False)
test_prepared.to_csv(PROCESSED_DIR / "test_prepared.csv", index=False)
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

for path in [
    INTERIM_DIR / "train_clean.csv",
    INTERIM_DIR / "test_clean.csv",
    PROCESSED_DIR / "train_prepared.csv",
    PROCESSED_DIR / "test_prepared.csv",
    MODELS_DIR / "preprocessor.joblib",
]:
    print(f"Guardado: {path}")

Guardado: ..\data\interim\train_clean.csv
Guardado: ..\data\interim\test_clean.csv
Guardado: ..\data\processed\train_prepared.csv
Guardado: ..\data\processed\test_prepared.csv
Guardado: ..\models\preprocessor.joblib
